# Spark Preparation
We check if we are in Google Colab.  If this is the case, install all necessary packages.

To run spark in Colab, we need to first install all the dependencies in Colab environment i.e. Apache Spark 3.3.2 with hadoop 3.3, Java 8 and Findspark to locate the spark in the system. The tools installation can be carried out inside the Jupyter Notebook of the Colab.
Learn more from [A Must-Read Guide on How to Work with PySpark on Google Colab for Data Scientists!](https://www.analyticsvidhya.com/blog/2020/11/a-must-read-guide-on-how-to-work-with-pyspark-on-google-colab-for-data-scientists/)

In [ ]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [ ]:
if IN_COLAB:
    !apt-get install openjdk-8-jdk-headless -qq > /dev/null
    !wget -q https://dlcdn.apache.org/spark/spark-3.3.2/spark-3.3.2-bin-hadoop3.tgz
    !tar xf spark-3.3.2-bin-hadoop3.tgz
    !mv spark-3.3.2-bin-hadoop3 spark
    !pip install -q findspark
    import os
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    os.environ["SPARK_HOME"] = "/content/spark"

# Start a Local Cluster

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, max

In [2]:
spark_url = 'local'
spark = SparkSession.builder\
    .master(spark_url)\
    .appName('Spark assignment')\
    .config('spark.ui.port','4040')\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/01 15:49:41 WARN Utils: Your hostname, Chatrins-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.105.1 instead (on interface bridge100)
26/04/01 15:49:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/01 15:49:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Spark Assignment

Based on the movie review dataset in 'netflix-rotten-tomatoes-metacritic-imdb.csv', answer the below questions.

**Note:** do not clean or remove missing data

In [3]:
df = spark.read.csv('netflix-rotten-tomatoes-metacritic-imdb.csv', header=True, inferSchema=True)

In [4]:
df.show(5)

+-------------------+--------------------+--------------------+----------------+---------------+----------------+--------------------+------------+---------------+--------------------+--------------------+-----------+----------+---------------------+----------------+---------------+--------------------+----------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+------------+
|              Title|               Genre|                Tags|       Languages|Series or Movie|Hidden Gem Score|Country Availability|     Runtime|       Director|              Writer|              Actors|View Rating|IMDb Score|Rotten Tomatoes Score|Metacritic Score|Awards Received|Awards Nominated For| Boxoffice|Release Date|Netflix Release Date|    Production House|        Netflix Link|           IMDb Link|             Summary|IMDb Votes|               Image|              

26/04/01 15:49:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## What is the maximum and average of the overall hidden gem score?

In [5]:
df.select(max(df['Hidden Gem Score']), avg(df['Hidden Gem Score'])).show()

+---------------------+---------------------+
|max(Hidden Gem Score)|avg(Hidden Gem Score)|
+---------------------+---------------------+
|                  9.8|    5.937551386501226|
+---------------------+---------------------+



## How many movies that are available in Korea?

In [6]:
from pyspark.sql.functions import split, explode

In [7]:
df_exploded = df.withColumn("Country", explode(split(df['Country Availability'], ',')))

In [8]:
country_count = df_exploded.groupBy('Country').count().sort("count", ascending=False)
country_count.show()

+--------------+-----+
|       Country|count|
+--------------+-----+
|United Kingdom| 6311|
|Czech Republic| 6237|
|         Japan| 6115|
|       Hungary| 5989|
|        Canada| 5930|
|     Singapore| 5876|
|      Thailand| 5876|
|         India| 5826|
|      Slovakia| 5820|
|     Australia| 5820|
|       Romania| 5759|
| United States| 5718|
|  South Africa| 5553|
|     Lithuania| 5541|
|       Germany| 5539|
|        Russia| 5491|
|   Switzerland| 5472|
|       Belgium| 5437|
|      Malaysia| 5394|
|       Iceland| 5364|
+--------------+-----+
only showing top 20 rows


In [9]:
country_count[country_count['Country']=='South Korea'].show()

+-----------+-----+
|    Country|count|
+-----------+-----+
|South Korea| 4845|
+-----------+-----+



In [34]:
lang_exploded = df.withColumn("lang", explode(split(df['Languages'], ', ')))
lang_count = lang_exploded.groupBy('lang').count().sort('count', ascending=False)
lang_count[lang_count['lang']=='Korean'].show()

+------+-----+
|  lang|count|
+------+-----+
|Korean|  735|
+------+-----+



## Which director has the highest average hidden gem score?

In [10]:
df.groupBy('Director').avg("Hidden Gem Score",).sort("avg(Hidden Gem Score)", ascending=False).show(1)

+-----------+---------------------+
|   Director|avg(Hidden Gem Score)|
+-----------+---------------------+
|Dorin Marcu|                  9.8|
+-----------+---------------------+
only showing top 1 row


## How many genres are there in the dataset?

In [21]:
explode_genre_df = df.withColumn("genre", explode(split(df["Genre"], ', ')))
explode_genre_df.show()

+-------------------+---------+--------------------+--------------------+---------------+----------------+--------------------+------------+---------------+--------------------+--------------------+-----------+----------+---------------------+----------------+---------------+--------------------+----------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+------------+
|              Title|    genre|                Tags|           Languages|Series or Movie|Hidden Gem Score|Country Availability|     Runtime|       Director|              Writer|              Actors|View Rating|IMDb Score|Rotten Tomatoes Score|Metacritic Score|Awards Received|Awards Nominated For| Boxoffice|Release Date|Netflix Release Date|    Production House|        Netflix Link|           IMDb Link|             Summary|IMDb Votes|               Image|              Poster|       

In [26]:
genre_count = explode_genre_df.groupBy('genre').count()
genre_count.show()

+-----------+-----+
|      genre|count|
+-----------+-----+
|      Crime| 1932|
|    Romance| 2445|
|   Thriller| 2739|
|  Adventure| 1809|
|      Drama| 6359|
|        War|  330|
|Documentary| 1030|
| Reality-TV|  191|
|     Family| 1433|
|    Fantasy| 1594|
|  Game-Show|   52|
|      Adult|   15|
|    History|  527|
|    Mystery| 1190|
|    Musical|  228|
|  Animation| 1665|
|      Music|  426|
|  Film-Noir|    2|
|     Horror| 1070|
|      Short|  422|
+-----------+-----+
only showing top 20 rows


In [27]:
genre_count.count()


28